# Domain-Specific Translation Accuracy
## Mini Project Notebook
**Case Study**: AI-Driven Real-Time Multilingual Communication

### Research Question
> *Can domain-specific adaptation improve translation accuracy for technical classroom content?*

### Languages: Hindi, Tamil, Telugu, Kannada
### Methods: Baseline NMT | Glossary Adaptation | Fine-Tuning

## 1. Setup & Imports

In [ ]:
import sys
sys.path.insert(0, '../src')

import pandas as pd
import matplotlib.pyplot as plt
import json
from pathlib import Path

DATA_DIR = Path('../data')
RESULTS_DIR = Path('../results')
RESULTS_DIR.mkdir(exist_ok=True)

print('Setup complete')

## 2. Explore the Dataset

In [ ]:
# Load the technical terms glossary
terms_df = pd.read_csv(DATA_DIR / 'technical_terms.csv')
print(f'Total technical terms: {len(terms_df)}')
print(f'Domains covered: {terms_df["domain"].unique()}')
terms_df.head(10)

In [ ]:
# Load test sentences
sentences_df = pd.read_csv(DATA_DIR / 'test_sentences.csv')
print(f'Total test sentences: {len(sentences_df)}')
sentences_df.head()

In [ ]:
# Domain distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

terms_df['domain'].value_counts().plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('Technical Terms by Domain')
axes[0].set_xlabel('Domain')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=0)

sentences_df['domain'].value_counts().plot(kind='bar', ax=axes[1], color='coral')
axes[1].set_title('Test Sentences by Domain')
axes[1].set_xlabel('Domain')
axes[1].set_ylabel('Count')
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'dataset_distribution.png', dpi=150)
plt.show()

## 3. Glossary Exploration

In [ ]:
# Show translations side-by-side for key terms
sample_terms = ['Neural Network', 'Transformer', 'Backpropagation', 
                'Loss Function', 'Attention Mechanism', 'Fine-tuning']

sample = terms_df[terms_df['term'].isin(sample_terms)][['term', 'hindi', 'tamil', 'telugu', 'kannada']]
sample.style.set_caption('Technical Term Translations')

## 4. Glossary Adaptation Demo (No Model Download Required)

In [ ]:
from glossary_adapter import GlossaryAdapter

# Demo: What glossary adaptation does
# Simulates NMT keeping English terms as-is (common behavior)
test_cases = [
    "The neural network uses backpropagation to update its weights.",
    "Gradient descent minimizes the loss function with a small learning rate.",
    "Overfitting is prevented by using dropout regularization.",
    "The transformer model applies the attention mechanism to all tokens.",
    "Transfer learning and fine-tuning improve domain-specific accuracy.",
]

for language in ['hindi', 'tamil', 'telugu', 'kannada']:
    adapter = GlossaryAdapter(str(DATA_DIR / 'technical_terms.csv'), language)
    print(f'\n--- {language.upper()} ---')
    for sentence in test_cases[:2]:  # Show 2 examples per language
        # Simulate baseline (NMT kept English terms)
        adapted, replacements = adapter.adapt(sentence, sentence)
        print(f'  EN:      {sentence[:70]}')
        print(f'  Adapted: {adapted[:70]}')
        print(f'  Terms replaced: {len(replacements)}')
        print()

## 5. Run Baseline Translation
(requires: `pip install transformers sentencepiece torch`)

In [ ]:
# Uncomment to run (downloads ~300MB of models)
# from baseline_translator import run_baseline_translation
# results = run_baseline_translation(
#     data_path=str(DATA_DIR / 'test_sentences.csv'),
#     output_path=str(RESULTS_DIR / 'baseline_translations.csv')
# )
# results.head()
print('Uncomment the cell above to run baseline translation')

## 6. Evaluate & Compare Results

In [ ]:
from evaluator import evaluate_all, save_evaluation_report, plot_comparison

# Load evaluation results if available
report_path = RESULTS_DIR / 'evaluation_report.json'
if report_path.exists():
    with open(report_path) as f:
        evaluation = json.load(f)
    
    # Show table
    rows = []
    for lang, methods in evaluation.items():
        for method, scores in methods.items():
            rows.append({
                'Language': lang.capitalize(),
                'Method': method.capitalize(),
                'BLEU Score': scores['avg_bleu'],
                'Term Accuracy': scores['avg_term_accuracy']
            })
    
    results_table = pd.DataFrame(rows)
    print(results_table.to_string(index=False))
    
    # Plot
    plot_comparison(evaluation, str(RESULTS_DIR / 'comparison_chart.png'))
    from IPython.display import Image
    Image(str(RESULTS_DIR / 'comparison_chart.png'))
else:
    print('No evaluation results yet. Run main.py first.')

## 7. Conclusion

| Method | Strengths | Weaknesses |
|--------|-----------|------------|
| **Baseline NMT** | Fast, no setup | Poor on technical terms |
| **Glossary Adaptation** | No training needed, instant | Limited to glossary terms |
| **Fine-Tuning** | Best overall accuracy | Requires training data & compute |

**Answer to Research Question**: Yes — domain-specific adaptation (both glossary and fine-tuning)
significantly improves translation accuracy for AI/Engineering technical content.